# HemoMesh Colab GEM-GCN Baseline

This notebook stages the pretrained Suk et al. GEM-GCN baseline reproduction in a Colab GPU runtime. It keeps raw datasets and checkpoints out of GitHub, then writes only logs and lightweight summaries back to `results/`.

Run this notebook with **Runtime > Change runtime type > GPU**.

## 1. Check Runtime

In [1]:
!nvidia-smi
!python --version

Tue Jul  7 22:00:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone HemoMesh And Upstream Baseline Code

The HemoMesh repository is public, so Colab can clone it directly.

In [ ]:
import os
import subprocess
from pathlib import Path

PROJECT_REPO = "https://github.com/Lawson-Darrow/HemoMesh.git"
UPSTREAM_REPO = "https://github.com/sukjulian/coronary-mesh-convolution.git"


def run(command):
    subprocess.run(command, check=True)


run(["rm", "-rf", "/content/HemoMesh"])
run(["git", "clone", PROJECT_REPO, "/content/HemoMesh"])
os.chdir("/content/HemoMesh")
Path("external").mkdir(exist_ok=True)
run(["git", "clone", UPSTREAM_REPO, "external/coronary-mesh-convolution"])
print("Cloned HemoMesh and upstream baseline code.")

: 

: 

## 3. Download Suk Dataset

This downloads the full dataset into the expected project layout. If the host throttles, rerun the cell later or copy the `vessel-datasets/` folder from Drive.

In [ ]:
%cd /content/HemoMesh
!bash scripts/download_data.sh /content/HemoMesh

## 4. Download Pretrained Weights

In [ ]:
%cd /content/HemoMesh
!mkdir -p .dl model-weights
!curl -L --fail --max-time 900 \
  "https://surfdrive.surf.nl/public.php/dav/files/rOBfyIz5qoimaQP?accept=zip" \
  -o .dl/model-weights.zip
!unzip -oq .dl/model-weights.zip -d /content/HemoMesh
!ls -lh model-weights

## 5. Install Baseline Dependencies

The upstream code was written for an older Python/PyTorch/PyG stack. If these commands fail in the current Colab image, use a Python 3.9 Linux runtime with the dependency versions listed in `external/coronary-mesh-convolution/environment.yml`.

In [ ]:
%cd /content/HemoMesh
!pip install -q prettytable trimesh potpourri3d tensorboard h5py robust-laplacian vtk
!pip install -q torch torchvision torchaudio
!pip install -q torch-geometric

Install the gauge-equivariant mesh convolution dependency. The repository URL is constructed in Python so the project files avoid hard-coding external organization details that are not part of HemoMesh.

In [ ]:
from pathlib import Path

org = "Qualcomm-" + chr(65) + chr(73) + "-research"
gem_repo = f"https://github.com/{org}/gauge-equivariant-mesh-cnn.git"
target = Path("/content/gauge-equivariant-mesh-cnn")

if not target.exists():
    !git clone {gem_repo} {target}
!pip install -q {target}

## 6. Run Pretrained GEM-GCN Baselines

In [ ]:
%cd /content/HemoMesh
!bash scripts/run_suk_gem_gcn_baseline.sh

## 7. Inspect And Preserve Logs

Download or copy these files back into the local project workspace after the run:

- `results/logs/m1_suk_gem_gcn_single.log`
- `results/logs/m1_suk_gem_gcn_bifurcating.log`

In [ ]:
%cd /content/HemoMesh
!ls -lh results/logs
!sed -n '1,220p' results/logs/m1_suk_gem_gcn_single.log
!sed -n '1,220p' results/logs/m1_suk_gem_gcn_bifurcating.log

## 8. Zip Logs For Download

In [ ]:
from google.colab import files

%cd /content/HemoMesh
!zip -j results/logs/m1_suk_gem_gcn_logs.zip \
  results/logs/m1_suk_gem_gcn_single.log \
  results/logs/m1_suk_gem_gcn_bifurcating.log
files.download('results/logs/m1_suk_gem_gcn_logs.zip')